In [49]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np

In [50]:
columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
           'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']

df = pd.read_csv('pima-indians-diabetes.csv', names=columns)

In [51]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [52]:
duplicates = df.duplicated()
print("Số dòng trùng lặp:", duplicates.sum())

df_cleaned = df.drop_duplicates()
print("Số dòng sau khi loại bỏ trùng lặp:", df_cleaned.shape)

Số dòng trùng lặp: 0
Số dòng sau khi loại bỏ trùng lặp: (768, 9)


In [53]:
physiological_ranges = {
    'Pregnancies': (0, 20),
    'Glucose': (40, 200),
    'BloodPressure': (20, 140),
    'SkinThickness': (5, 99),
    'Insulin': (15, 846),
    'BMI': (15, 60),
    'DiabetesPedigreeFunction': (0.0, 2.5),
    'Age': (10, 100)
}

In [54]:
def detect_physiological_errors(df, physiological_ranges):
    if not isinstance(df, pd.DataFrame):
        raise TypeError('df must be a pandas DataFrame')
    if not isinstance(physiological_ranges, dict):
        raise TypeError('physiological_ranges must be a dict')
    errors = {}
    for col, valid_range in physiological_ranges.items():
        if col not in df.columns:
            continue
        if not (isinstance(valid_range, (tuple, list)) and len(valid_range) == 2):
            raise ValueError(f'Range for {col} must be tuple/list (min, max)')
        min_val, max_val = valid_range
        col_values = df[col]
        invalid_mask = col_values.notna() & ((col_values < min_val) | (col_values > max_val))
        invalid_values = col_values[invalid_mask]
        if not invalid_values.empty:
            errors[col] = {
                'count': int(invalid_values.shape[0]),
                'min_actual': float(col_values.min()) if pd.notna(col_values.min()) else None,
                'max_actual': float(col_values.max()) if pd.notna(col_values.max()) else None,
                'problem_values': sorted(set(invalid_values.tolist()))
            }
    return errors

In [55]:
errors = detect_physiological_errors(df, physiological_ranges)
print("=== Dữ liệu lỗi phát hiện ===")
for col, info in errors.items():
        print(f"{col}: {info['count']} giá trị lỗi")
        print(f" -Range thực tế: {info['min_actual']} - {info['max_actual']}")
        print(f" -Giá trị có vấn đề: {info['problem_values']}")

=== Dữ liệu lỗi phát hiện ===
Glucose: 5 giá trị lỗi
 -Range thực tế: 0.0 - 199.0
 -Giá trị có vấn đề: [0]
BloodPressure: 35 giá trị lỗi
 -Range thực tế: 0.0 - 122.0
 -Giá trị có vấn đề: [0]
SkinThickness: 227 giá trị lỗi
 -Range thực tế: 0.0 - 99.0
 -Giá trị có vấn đề: [0]
Insulin: 375 giá trị lỗi
 -Range thực tế: 0.0 - 846.0
 -Giá trị có vấn đề: [0, 14]
BMI: 12 giá trị lỗi
 -Range thực tế: 0.0 - 67.1
 -Giá trị có vấn đề: [0.0, 67.1]


In [56]:

def handle_zero_values(df):
    zero_sensitive_columns = ['Insulin', 'SkinThickness']
    for col in zero_sensitive_columns:
        zero_mask = df[col] == 0
        if zero_mask.any():
            print(f"Phát hiện {zero_mask.sum()} giá trị 0 trong {col}")
            median_val = df[col][df[col] > 0].median()
            df.loc[zero_mask, col] = median_val
            print(f" -> Đã thay thế bằng median: {median_val:.2f}")
    return df
df_final = handle_zero_values(df_cleaned)

Phát hiện 374 giá trị 0 trong Insulin
 -> Đã thay thế bằng median: 125.00
Phát hiện 227 giá trị 0 trong SkinThickness
 -> Đã thay thế bằng median: 29.00


In [57]:
for col, (min_val, max_val) in physiological_ranges.items():
    df_final = df_final[(df_final[col] >= min_val) & (df_final[col] <= max_val)]

print("=== KẾT QUẢ SAU XỬ LÝ ===")
print(f"Dữ liệu gốc: {len(df)} dòng")
print(f"Sau xử lý: {len(df_final)} dòng")
print(f"Tỷ lệ giữ lại: {len(df_final)/len(df)*100:.1f}%")

final_errors = detect_physiological_errors(df_final, physiological_ranges)

if not final_errors:
    print("Tất cả dữ liệu đều trong ngưỡng sinh lý hợp lý")
else:
    print("Vẫn còn dữ liệu lỗi:")
    for col, info in final_errors.items():
        print(f" - {col}: {info['count']} lỗi")

=== KẾT QUẢ SAU XỬ LÝ ===
Dữ liệu gốc: 768 dòng
Sau xử lý: 722 dòng
Tỷ lệ giữ lại: 94.0%
Tất cả dữ liệu đều trong ngưỡng sinh lý hợp lý


In [58]:
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


In [59]:
df_final.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,125,33.6,0.627,50,1
1,1,85,66,29,125,26.6,0.351,31,0
2,8,183,64,29,125,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [60]:
import pandas as pd

# dataset đã xử lý ở chương 3
df = df_final.copy()

# biến đầu vào
X = df.drop("Outcome", axis=1)

# biến đầu ra
y = df["Outcome"]

print("Kích thước dữ liệu:")
print("X:", X.shape)
print("y:", y.shape)

Kích thước dữ liệu:
X: (722, 8)
y: (722,)


In [61]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print("Kích thước tập dữ liệu:")
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Kích thước tập dữ liệu:
Train: (541, 8)
Test: (181, 8)


In [62]:
# dữ liệu gốc không chuẩn hóa
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()

print("RAW DATA sample:")
X_train_raw.head()

RAW DATA sample:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
21,8,99,84,29,125,35.4,0.388,50
683,4,125,80,29,125,32.3,0.536,27
455,14,175,62,30,125,33.6,0.212,38
501,3,84,72,32,125,37.2,0.267,28
91,4,123,80,15,176,32.0,0.443,34


In [63]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()

X_train_minmax = minmax_scaler.fit_transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

print("MinMax sample:")
print(X_train_minmax[:5])

MinMax sample:
[[0.47058824 0.35483871 0.58695652 0.23913043 0.13237064 0.4398977
  0.13771657 0.60416667]
 [0.23529412 0.52258065 0.54347826 0.23913043 0.13237064 0.36061381
  0.20346513 0.125     ]
 [0.82352941 0.84516129 0.34782609 0.25       0.13237064 0.39386189
  0.0595291  0.35416667]
 [0.17647059 0.25806452 0.45652174 0.27173913 0.13237064 0.4859335
  0.08396268 0.14583333]
 [0.23529412 0.50967742 0.54347826 0.08695652 0.19374248 0.35294118
  0.16215016 0.27083333]]


In [64]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()

X_train_std = standard_scaler.fit_transform(X_train)
X_test_std = standard_scaler.transform(X_test)

print("Standard sample:")
print(X_train_std[:5])

Standard sample:
[[ 1.2845001  -0.75904208  0.96956323 -0.02419429 -0.2035148   0.42392476
  -0.26487011  1.45809054]
 [ 0.07384757  0.10281503  0.64346208 -0.02419429 -0.2035148  -0.0317047
   0.18479637 -0.5440084 ]
 [ 3.10047889  1.76023256 -0.82399312  0.08579823 -0.2035148   0.15936572
  -0.79960862  0.41351718]
 [-0.22881557 -1.25626734 -0.00874023  0.30578327 -0.2035148   0.68848381
  -0.63250284 -0.45696062]
 [ 0.07384757  0.03651833  0.64346208 -1.56408958  0.34850809 -0.07579788
  -0.09776433  0.06532606]]


# Xây dựng và huấn luyện mô hình

### 1. Logistic Regression

In [65]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# dùng dữ liệu đã chuẩn hóa (StandardScaler)
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train_std, y_train)

y_pred_log = log_model.predict(X_test_std)

print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

=== Logistic Regression ===
Accuracy: 0.7845303867403315
              precision    recall  f1-score   support

           0       0.83      0.85      0.84       123
           1       0.67      0.64      0.65        58

    accuracy                           0.78       181
   macro avg       0.75      0.75      0.75       181
weighted avg       0.78      0.78      0.78       181



### 2. K-Nearest Neighbors

In [66]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_std, y_train)

y_pred_knn = knn_model.predict(X_test_std)

print("=== KNN ===")
print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

=== KNN ===
Accuracy: 0.7790055248618785
              precision    recall  f1-score   support

           0       0.84      0.83      0.84       123
           1       0.65      0.67      0.66        58

    accuracy                           0.78       181
   macro avg       0.75      0.75      0.75       181
weighted avg       0.78      0.78      0.78       181



### 3. Decision Tree

In [67]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(random_state=42)

tree_model.fit(X_train_raw, y_train)

y_pred_tree = tree_model.predict(X_test_raw)

print("=== Decision Tree ===")
print("Accuracy:", accuracy_score(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

=== Decision Tree ===
Accuracy: 0.7458563535911602
              precision    recall  f1-score   support

           0       0.82      0.80      0.81       123
           1       0.60      0.64      0.62        58

    accuracy                           0.75       181
   macro avg       0.71      0.72      0.71       181
weighted avg       0.75      0.75      0.75       181



### 4. Random Forest

In [68]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_raw, y_train)

y_pred_rf = rf_model.predict(X_test_raw)

print("=== Random Forest ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

=== Random Forest ===
Accuracy: 0.7790055248618785
              precision    recall  f1-score   support

           0       0.84      0.84      0.84       123
           1       0.66      0.66      0.66        58

    accuracy                           0.78       181
   macro avg       0.75      0.75      0.75       181
weighted avg       0.78      0.78      0.78       181



In [69]:
results = {
    "Logistic Regression": accuracy_score(y_test, y_pred_log),
    "KNN": accuracy_score(y_test, y_pred_knn),
    "Decision Tree": accuracy_score(y_test, y_pred_tree),
    "Random Forest": accuracy_score(y_test, y_pred_rf),
}

print("\n=== SO SÁNH ACCURACY ===")
for model, acc in results.items():
    print(f"{model}: {acc:.4f}")


=== SO SÁNH ACCURACY ===
Logistic Regression: 0.7845
KNN: 0.7790
Decision Tree: 0.7459
Random Forest: 0.7790


In [70]:
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Tạo dictionary lưu kết quả
results = {
    "Logistic Regression": [
        accuracy_score(y_test, y_pred_log),
        precision_score(y_test, y_pred_log),
        recall_score(y_test, y_pred_log),
        f1_score(y_test, y_pred_log)
    ],
    
    "KNN": [
        accuracy_score(y_test, y_pred_knn),
        precision_score(y_test, y_pred_knn),
        recall_score(y_test, y_pred_knn),
        f1_score(y_test, y_pred_knn)
    ],
    
    "Decision Tree": [
        accuracy_score(y_test, y_pred_tree),
        precision_score(y_test, y_pred_tree),
        recall_score(y_test, y_pred_tree),
        f1_score(y_test, y_pred_tree)
    ],
    
    "Random Forest": [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf)
    ]
}

# Tạo DataFrame
df_results = pd.DataFrame(
    results,
    index=["Accuracy", "Precision", "Recall", "F1-score"]
).T   # .T để chuyển model thành dòng

# Hiển thị bảng
# display(df_results)
# display(df_results.style.background_gradient(cmap='Blues'))
display(
    df_results.style
    .background_gradient(cmap='Blues')
    .highlight_max(color='green')
)

,Accuracy,Precision,Recall,F1-score
Logistic Regression,0.784530,0.672727,0.637931,0.654867
KNN,0.779006,0.650000,0.672414,0.661017
Decision Tree,0.745856,0.596774,0.637931,0.616667
Random Forest,0.779006,0.655172,0.655172,0.655172


### Model tốt nhất theo từng độ đánh giá mô hình

In [71]:
best_accuracy_model = df_results['Accuracy'].idxmax()
print("Model tốt nhất theo Accuracy:", best_accuracy_model)
best_precision_model = df_results['Precision'].idxmax()
print("Model tốt nhất theo Precision:", best_precision_model)
best_recall_model = df_results['Recall'].idxmax()
print("Model tốt nhất theo Recall:", best_recall_model)
best_f1_model = df_results['F1-score'].idxmax()
print("Model tốt nhất theo F1-score:", best_f1_model)

Model tốt nhất theo Accuracy: Logistic Regression
Model tốt nhất theo Precision: Logistic Regression
Model tốt nhất theo Recall: KNN
Model tốt nhất theo F1-score: KNN


### Huẩn luyện mô hình trên dữ liệu thô

In [ ]:
# 1. IMPORT THƯ VIỆN
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 2. LOAD DỮ LIỆU
df = pd.read_csv('pima-indians-diabetes.csv', names=columns)

# 3. TIỀN XỬ LÝ DỮ LIỆU
# 1. Chuẩn hóa tên cột (PHẢI làm đầu tiên)
df.columns = df.columns.str.strip().str.lower()

# 2. CHỈ thay 0 ở các cột feature (KHÔNG đụng vào outcome)
cols = ['glucose', 'bloodpressure', 'skinthickness', 'insulin', 'bmi']

df[cols] = df[cols].replace(0, np.nan)

# 3. Fill NaN cho các cột này
df[cols] = df[cols].fillna(df[cols].median())

# 4. Kiểm tra
print("Số NaN còn lại:", df.isnull().sum().sum())
print("Phân bố nhãn:\n", df['outcome'].value_counts().to_string())

# 4. TÁCH X, y
X = df.drop("outcome", axis=1)   # chú ý: lowercase
y = df["outcome"]

# 5. CHIA TRAIN / TEST
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Phân bố nhãn trong train:\n", y_train.value_counts().to_string())
print("Phân bố nhãn trong test:\n", y_test.value_counts().to_string())

# 6. CHUẨN HÓA (cho Logistic, KNN)
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train_raw)
X_test_std = scaler.transform(X_test_raw)

# 7. HUẤN LUYỆN MÔ HÌNH
# Logistic Regression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_std, y_train)

# KNN
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_std, y_train)

# Decision Tree
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train_raw, y_train)

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_raw, y_train)

# 8. DỰ ĐOÁN
y_pred_log = log_model.predict(X_test_std)
y_pred_knn = knn_model.predict(X_test_std)
y_pred_tree = tree_model.predict(X_test_raw)
y_pred_rf = rf_model.predict(X_test_raw)

# 9. ĐÁNH GIÁ
def evaluate(y_true, y_pred):
    return [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred),
        recall_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    ]

results = {
    "Logistic Regression": evaluate(y_test, y_pred_log),
    "KNN": evaluate(y_test, y_pred_knn),
    "Decision Tree": evaluate(y_test, y_pred_tree),
    "Random Forest": evaluate(y_test, y_pred_rf),
}

df_results = pd.DataFrame(
    results,
    index=["Accuracy", "Precision", "Recall", "F1-score"]
).T

from IPython.display import display
# display(df_results.style.background_gradient(cmap='Blues'))
display(
    df_results.style
    .background_gradient(cmap='Blues')
    .highlight_max(color='green')
)

Số NaN còn lại: 0
Phân bố nhãn:
 outcome
0    500
1    268
Phân bố nhãn trong train:
 outcome
0    401
1    213
Phân bố nhãn trong test:
 outcome
0    99
1    55


,Accuracy,Precision,Recall,F1-score
Logistic Regression,0.753247,0.666667,0.618182,0.641509
KNN,0.720779,0.596774,0.672727,0.632479
Decision Tree,0.720779,0.603448,0.636364,0.619469
Random Forest,0.746753,0.637931,0.672727,0.654867


### Huấn luyện mô hình trên dữ liệu Min - Max

In [98]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()

X_train_mm = minmax.fit_transform(X_train_raw)
X_test_mm = minmax.transform(X_test_raw)

# Logistic Regression (Min-Max)
log_model_mm = LogisticRegression(max_iter=1000)
log_model_mm.fit(X_train_mm, y_train)

# KNN (Min-Max)
knn_model_mm = KNeighborsClassifier(n_neighbors=5)
knn_model_mm.fit(X_train_mm, y_train)

y_pred_log_mm = log_model_mm.predict(X_test_mm)
y_pred_knn_mm = knn_model_mm.predict(X_test_mm)

results_mm = {
    "Logistic (MinMax)": evaluate(y_test, y_pred_log_mm),
    "KNN (MinMax)": evaluate(y_test, y_pred_knn_mm),
}

df_results_mm = pd.DataFrame(
    results_mm,
    index=["Accuracy", "Precision", "Recall", "F1-score"]
).T

from IPython.display import display
display(df_results_mm.style.background_gradient(cmap='Greens'))

,Accuracy,Precision,Recall,F1-score
Logistic (MinMax),0.766234,0.702128,0.600000,0.647059
KNN (MinMax),0.733766,0.616667,0.672727,0.643478


### Huấn luyện mô hình trên dữ liệu Standard

In [99]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_std = scaler.fit_transform(X_train_raw)
X_test_std = scaler.transform(X_test_raw)

# Logistic Regression
log_model_std = LogisticRegression(max_iter=1000)
log_model_std.fit(X_train_std, y_train)

# KNN
knn_model_std = KNeighborsClassifier(n_neighbors=5)
knn_model_std.fit(X_train_std, y_train)

y_pred_log_std = log_model_std.predict(X_test_std)
y_pred_knn_std = knn_model_std.predict(X_test_std)

results_std = {
    "Logistic (Standard)": evaluate(y_test, y_pred_log_std),
    "KNN (Standard)": evaluate(y_test, y_pred_knn_std),
}

df_results_std = pd.DataFrame(
    results_std,
    index=["Accuracy", "Precision", "Recall", "F1-score"]
).T

from IPython.display import display
display(df_results_std.style.background_gradient(cmap='Oranges'))

,Accuracy,Precision,Recall,F1-score
Logistic (Standard),0.753247,0.666667,0.618182,0.641509
KNN (Standard),0.720779,0.596774,0.672727,0.632479


In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]

df_all = pd.concat([df_results, df_results_std, df_results_mm])

for metric in metrics:
    best_model = df_all[metric].idxmax()
    best_value = df_all[metric].max()
    
    print(f"{metric}:")
    print(f" -> Model tốt nhất: {best_model}")
    print(f" -> Giá trị: {best_value:.4f}\n")

Accuracy:
 -> Model tốt nhất: Logistic (MinMax)
 -> Giá trị: 0.7662

Precision:
 -> Model tốt nhất: Logistic (MinMax)
 -> Giá trị: 0.7021

Recall:
 -> Model tốt nhất: KNN
 -> Giá trị: 0.6727

F1-score:
 -> Model tốt nhất: Random Forest
 -> Giá trị: 0.6549



In [126]:
ranked = df_all.sort_values(by="Accuracy", ascending=False)

ranked["Rank"] = range(1, len(ranked) + 1)

display(
    ranked[["Rank", "Accuracy"]].style
    .background_gradient(cmap='Blues')
)

,Rank,Accuracy
Logistic (MinMax),1,0.766234
Logistic Regression,2,0.753247
Logistic (Standard),3,0.753247
Random Forest,4,0.746753
KNN (MinMax),5,0.733766
KNN,6,0.720779
Decision Tree,7,0.720779
KNN (Standard),8,0.720779


In [127]:
ranked = df_all.sort_values(by="Precision", ascending=False)

ranked["Rank"] = range(1, len(ranked) + 1)

display(
    ranked[["Rank", "Precision"]].style
    .background_gradient(cmap='Blues')
)

,Rank,Precision
Logistic (MinMax),1,0.702128
Logistic Regression,2,0.666667
Logistic (Standard),3,0.666667
Random Forest,4,0.637931
KNN (MinMax),5,0.616667
Decision Tree,6,0.603448
KNN,7,0.596774
KNN (Standard),8,0.596774


In [128]:
ranked = df_all.sort_values(by="Recall", ascending=False)

ranked["Rank"] = range(1, len(ranked) + 1)

display(
    ranked[["Rank", "Recall"]].style
    .background_gradient(cmap='Blues')
)

,Rank,Recall
KNN,1,0.672727
Random Forest,2,0.672727
KNN (MinMax),3,0.672727
KNN (Standard),4,0.672727
Decision Tree,5,0.636364
Logistic Regression,6,0.618182
Logistic (Standard),7,0.618182
Logistic (MinMax),8,0.600000


In [125]:
ranked = df_all.sort_values(by="F1-score", ascending=False)

ranked["Rank"] = range(1, len(ranked) + 1)

display(
    ranked[["Rank", "F1-score"]].style
    .background_gradient(cmap='Blues')
)

,Rank,F1-score
Random Forest,1,0.654867
Logistic (MinMax),2,0.647059
KNN (MinMax),3,0.643478
Logistic Regression,4,0.641509
Logistic (Standard),5,0.641509
KNN,6,0.632479
KNN (Standard),7,0.632479
Decision Tree,8,0.619469


### Kết luận

In [115]:
best_acc = df_all['Accuracy'].idxmax()
best_precision = df_all['Precision'].idxmax()
best_f1 = df_all['F1-score'].idxmax()
best_recall = df_all['Recall'].idxmax()

print("=== KẾT LUẬN ===")
print(f"- Model tốt nhất theo Accuracy: {best_acc}")
print(f"- Model tốt nhất theo Precision: {best_precision}")
print(f"- Model tốt nhất theo F1-score: {best_f1}")
print(f"- Model tốt nhất theo Recall): {best_recall}")

=== KẾT LUẬN ===
- Model tốt nhất theo Accuracy: Logistic (MinMax)
- Model tốt nhất theo Precision: Logistic (MinMax)
- Model tốt nhất theo F1-score: Random Forest
- Model tốt nhất theo Recall): KNN
